In [ ]:
import pandas

from plotly import graph_objects
from plotly import offline as plotly
from plotly import express as plotly_express

In [ ]:
personal_food_expenditure = pandas.read_csv("personal_food_expenditures.csv")
personal_food_expenditure

In [ ]:
joint_food_expenditure = pandas.read_csv("joint_food_expenditures.csv")
joint_food_expenditure

In [ ]:
# Plot the sum of expenditures in each of personal and joint food expenditures
# by month

data = []

for _, row in personal_food_expenditure.iterrows():
    data.append({
        "month": row["month"],
        "expenditure": row["expenditure"],
        "source": "personal",
        "category_name": row["category_name"]
    })

for _, row in joint_food_expenditure.iterrows():
    data.append({
        "month": row["month"],
        "expenditure": row["expenditure"] / 2,
        "source": "joint",
        "category_name": row["category_name"]
    })

# Reset index after groupby to convert MultiIndex back to regular columns
df = pandas.DataFrame.from_records(data)

# df = df[df["category_name"] == "Eating Out"]

df = df.groupby(["month", "source"]).sum().reset_index()

# Plot two traces, one for personal and one for joint

figure = plotly_express.line(df, x="month", y="expenditure", color="source")

figure.show()


In [ ]:
# Calculate the sum across joint and personal, and plot just that

df = pandas.DataFrame.from_records(data).groupby(["month"]).sum().reset_index()

figure = plotly_express.line(df, x="month", y="expenditure")

figure.show()

In [ ]:
df.mean()

In [ ]:
import scipy
from scipy import stats

# Calculate a t-test between the month sums before June 2024 and after

df["State"] = df["month"].apply(lambda x: "Together" if x < "2024-06-01" else "Broken Up")

df_before = df[df["month"] < "2024-06-01"]
df_after = df[df["month"] >= "2024-06-01"]

t_test = scipy.stats.ttest_ind(df_before["expenditure"], df_after["expenditure"])

figure = plotly_express.box(df, color="State", y="expenditure", points="all")

# Add a title and make the points more transparent and the background white

figure.update_layout(
    title="Food Expenditure per Month",
    xaxis_title="Did we break up?",
    yaxis_title="Expenditure",
    showlegend=True,
    legend_title="Did we break up?"
)

# Add the p-value to the plot above a line that is 100 units above the highest expenditure
# and extends between the two boxes

# Get the maximum expenditure value to position the significance lines
max_expenditure = df["expenditure"].max()
line_height = max_expenditure + 100

# Add horizontal line connecting the two box centers
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,  # Start at "Broken Up" box center
    x1=0.175, y1=line_height,  # End at "Together" box center
    line=dict(color="black", width=2)
)

# Add vertical lines extending down from horizontal line
# Left vertical line (Broken Up)
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,
    x1=-0.175, y1=line_height - 30,
    line=dict(color="black", width=2)
)

# Right vertical line (Together)
figure.add_shape(
    type="line",
    x0=0.175, y0=line_height,
    x1=0.175, y1=line_height - 30,
    line=dict(color="black", width=2)
)

# Add p-value annotation above the horizontal line
p_value = t_test.pvalue
figure.add_annotation(
    x=0,  # Center between the two boxes
    y=line_height + 20,
    text=f"p = {p_value:.2e}",
    showarrow=False,
    font=dict(size=12, color="black"),
    xanchor="center"
)

# Make the background transparent

figure.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=800,
)

# Make the y-axis in dollars

figure.update_yaxes(tickprefix="$")

figure.show()

In [ ]:
food_time_usage = pandas.read_csv("food_time_usage.csv")
food_time_usage

In [ ]:
data = []

for row_index, row in food_time_usage.iterrows():

    x = row["hours_per_week"]
    x1 = x * 4.3
    text = str(row["month"])

    expenditure = 0

    for label, df in [("joint", joint_food_expenditure), ("personal", personal_food_expenditure)]:

        for _, expenditure_row in df.iterrows():

            if expenditure_row["month"] != row["month"]:
                continue

            if label == "joint":
                expenditure += expenditure_row["expenditure"] / 2
            else:
                expenditure += expenditure_row["expenditure"]

    if row["month"] < "2024-06-01":
        label = "Together"
    else:
        label = "Broken Up"

    data.append({
        "hours_per_week": x,
        "hours_per_month": x1,
        "text": text,
        "expenditure": expenditure,
        "label": label
    })

df = pandas.DataFrame.from_records(data)

In [ ]:
# Make a plot of time vs expenditure

figure = plotly_express.scatter(df, x="hours_per_month", y="expenditure", hover_data="text")

figure.update_layout({
    "width": 600,
    "height": 600,
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "xaxis_title": "Time Spent Procuring, Preparing, and Consuming Food(Hours)",
    "yaxis_title": "Monthly Food Expenditure"
})

# Format y-axis as dollars
figure.update_yaxes(tickprefix="$")

plotly.iplot(figure)

In [ ]:
from scipy.stats import linregress

In [ ]:
label = "Together"

linregress(x=df[df["label"] == label]["hours_per_month"], y=df[df["label"] == label]["expenditure"])

In [ ]:
label = "Broken Up"

linregress(x=df[df["label"] == label]["hours_per_month"], y=df[df["label"] == label]["expenditure"])

Conclusion: for every hour per month I spend on preparing food, I save $10.70 on groceries

In [ ]:
import scipy
from scipy import stats

# Calculate a t-test between the month sums before June 2024 and after

df_together = df[df["label"] == "Together"]
df_broken_up = df[df["label"] == "Broken Up"]

t_test = scipy.stats.ttest_ind(df_together["expenditure"], df_broken_up["expenditure"])

figure = plotly_express.box(df, color="label", y="expenditure", points="all")

# Add a title and make the points more transparent and the background white

figure.update_layout(
    title="Food Expenditure per Month",
    xaxis_title="Did we break up?",
    yaxis_title="Expenditure",
    showlegend=True,
    legend_title="Did we break up?"
)

# Add the p-value to the plot above a line that is 100 units above the highest expenditure
# and extends between the two boxes

# Get the maximum expenditure value to position the significance lines
max_expenditure = df["expenditure"].max()
line_height = max_expenditure + 100

# Add horizontal line connecting the two box centers
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,  # Start at "Broken Up" box center
    x1=0.175, y1=line_height,  # End at "Together" box center
    line=dict(color="black", width=2)
)

# Add vertical lines extending down from horizontal line
# Left vertical line (Broken Up)
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,
    x1=-0.175, y1=line_height - 30,
    line=dict(color="black", width=2)
)

# Right vertical line (Together)
figure.add_shape(
    type="line",
    x0=0.175, y0=line_height,
    x1=0.175, y1=line_height - 30,
    line=dict(color="black", width=2)
)

# Add p-value annotation above the horizontal line
p_value = t_test.pvalue
figure.add_annotation(
    x=0,  # Center between the two boxes
    y=line_height + 20,
    text=f"p = {p_value:.2e}",
    showarrow=False,
    font=dict(size=12, color="black"),
    xanchor="center"
)

# Make the background transparent

figure.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=800,
)

# Make the y-axis in dollars

figure.update_yaxes(tickprefix="$")

figure.show()

In [ ]:
import scipy
from scipy import stats

# Calculate a t-test between the month sums before June 2024 and after

df_together = df[df["label"] == "Together"]
df_broken_up = df[df["label"] == "Broken Up"]

t_test = scipy.stats.ttest_ind(df_together["hours_per_month"], df_broken_up["hours_per_month"])

figure = plotly_express.box(df, color="label", y="hours_per_month", points="all")

# Add a title and make the points more transparent and the background white

figure.update_layout(
    title="Time Spent on Food per Month",
    xaxis_title="Did we break up?",
    yaxis_title="Hours per Month",
    showlegend=True,
    legend_title="Did we break up?"
)

# Add the p-value to the plot above a line that is 100 units above the highest expenditure
# and extends between the two boxes

# Get the maximum hours_per_month value to position the significance lines
max_hours_per_month = df["hours_per_month"].max()
line_height = max_hours_per_month + 10

# Add horizontal line connecting the two box centers
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,  # Start at "Broken Up" box center
    x1=0.175, y1=line_height,  # End at "Together" box center
    line=dict(color="black", width=2)
)

# Add vertical lines extending down from horizontal line
# Left vertical line (Broken Up)
figure.add_shape(
    type="line",
    x0=-0.175, y0=line_height,
    x1=-0.175, y1=line_height - 5,
    line=dict(color="black", width=2)
)

# Right vertical line (Together)
figure.add_shape(
    type="line",
    x0=0.175, y0=line_height,
    x1=0.175, y1=line_height - 5,
    line=dict(color="black", width=2)
)

# Add p-value annotation above the horizontal line
p_value = t_test.pvalue
figure.add_annotation(
    x=0,  # Center between the two boxes
    y=line_height + 20,
    text=f"p = {p_value:.2e}",
    showarrow=False,
    font=dict(size=12, color="black"),
    xanchor="center"
)

# Make the background transparent

figure.update_layout(
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    width=800,
)

figure.show()